In [0]:
%run ./00_common_dq_functions

In [0]:
from pyspark.sql.functions import col, to_timestamp


In [0]:
spark.sql("USE CATALOG pharma_medallion_v2")
spark.sql("USE SCHEMA dev_bronze")

DataFrame[]

In [0]:
bronze_db = "dev_bronze"
silver_db = "dev_silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_db}")

DataFrame[]

In [0]:
qc_bz = spark.table(f"{bronze_db}.fact_qc_tests_src_bz")

qc_std = (
    qc_bz
    .withColumn("batch_id", col("batch_id").cast("string"))
    .withColumn("test_name", col("test_name").cast("string"))
    .withColumn("test_time", to_timestamp(col("test_time")))
)

# Standardize numeric columns if they exist
for ncol in ["test_value", "result_value", "upper_limit", "lower_limit"]:
    if ncol in qc_std.columns:
        qc_std = qc_std.withColumn(ncol, col(ncol).cast("double"))

qc_good1, qc_bad1 = dq_not_null(
    qc_std,
    ["batch_id", "test_name", "test_time"]
)

# ============================================================
# 4) DQ Rule 2 – Deduplication
# ============================================================

qc_good2, qc_bad2 = dq_dedup(
    qc_good1,
    ["batch_id", "test_name", "test_time"]
)

# ============================================================
# 5) DQ Rule 3 – Timeliness (no future timestamps)
# ============================================================

qc_good3, qc_bad3 = dq_timeliness_no_future(
    qc_good2,
    "test_time"
)

# ============================================================
# 6) DQ Rule 4 – FK check (batch_id exists in batch_header)
# ============================================================

batch_header_sl = spark.table(f"{silver_db}.batch_header_sl")

qc_sl, qc_bad4 = dq_fk_exists(
    qc_good3,
    "batch_id",
    batch_header_sl,
    "batch_id"
)

# ============================================================
# 7) Union all bad rows & write quarantine
# ============================================================

qc_bad_all = dq_union_bad(
    qc_bad1,
    qc_bad2,
    qc_bad3,
    qc_bad4
)

dq_write_quarantine(
    qc_bad_all,
    f"{silver_db}.dq_quarantine_fact_qc_tests_src"
)

# ============================================================
# 8) Write Silver QC Tests fact table
# ============================================================

qc_sl.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(f"{silver_db}.fact_qc_tests_src_sl")


In [0]:
print("fact_qc_tests_src_sl:", spark.table(f"{silver_db}.fact_qc_tests_src_sl").count())
print("dq_quarantine_fact_qc_tests_src:",
      spark.table(f"{silver_db}.dq_quarantine_fact_qc_tests_src").count())

fact_qc_tests_src_sl: 119
dq_quarantine_fact_qc_tests_src: 4
